# 01 – Data Factory: Q&A Dataset Generation

**Operation Ledger-Mind: The Financial Intelligence**

**Goal:** Transform Uber's 2024 Annual Report PDF into structured instruction dataset.

**Pipeline:**
1. Load and clean PDF
2. Chunk into 1500-character segments
3. Generate 3 Q&A pairs per chunk using SINGLE efficient LLM (Groq/Gemini)
4. Use ALL 482 chunks for maximum dataset
5. Split into train (80%) and test (20%) sets

**Configuration:**
- LLM: Groq (free, fast, 30k tokens/min) OR Gemini (free)
- Total Q&A pairs: ~1,446 (482 chunks × 3 pairs)
- Estimated time: ~30-45 minutes

In [1]:
# Setup: Config & Services
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
from tqdm import tqdm

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Load environment
load_dotenv()

# Import from src
from src.services.llm_services import (
    load_config,
    get_llm,
    validate_api_keys,
    print_config_summary
)

# Load config
config = load_config("../src/config/config.yaml")
validate_api_keys(config, verbose=True)
print_config_summary(config)

Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


In [2]:
# Initialize LLM
llm = get_llm(config)
print(f"LLM initialized")

# Test connection
print("\nTesting API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"API verified: {test_msg[:50]}")
except Exception as e:
    print(f"API test failed: {e}")

LLM initialized

Testing API connection...
API verified: API working!


## Step 1: PDF Processing

In [3]:
import fitz  # PyMuPDF
import re

def clean_text(text: str) -> str:
    """Clean extracted PDF text."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\x00', '', text)
    text = text.replace('ﬁ', 'fi').replace('ﬂ', 'fl')
    return text.strip()

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from PDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return clean_text(text)

def chunk_text(text: str, chunk_size: int = 1500, overlap: int = 150):
    chunks = []
    start = 0
    chunk_id = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + chunk_size, text_length)

        if end < text_length:
            for punct in ['.', '!', '?']:
                pos = text.rfind(punct, max(start, end - 200), end)
                if pos != -1:
                    end = pos + 1
                    break

        chunk = text[start:end].strip()

        if not chunk:
            break

        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk,
            "start": start,
            "end": end,
            "length": len(chunk),
        })

        chunk_id += 1

        if end >= text_length:
            break

        start = max(end - overlap, start + 1)

    return chunks

print("PDF utilities ready")

PDF utilities ready


In [4]:
# Load and chunk PDF
pdf_path = Path(config['data_root']) / "2024-Annual-Report.pdf"

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

print(f"Extracting text from: {pdf_path.name}")
full_text = extract_text_from_pdf(str(pdf_path))
print(f"Extracted {len(full_text):,} characters")

print(f"\nChunking with size=1500, overlap=150")
chunks = chunk_text(full_text, 1500, 150)
print(f"Created {len(chunks)} chunks")
print(f"   Average length: {sum(c['length'] for c in chunks) / len(chunks):.0f} chars")

Extracting text from: 2024-Annual-Report.pdf
Extracted 620,206 characters

Chunking with size=1500, overlap=150
Created 482 chunks
   Average length: 1436 chars


## Step 2: Q&A Generation

In [5]:
def generate_qa_pairs(chunk_text: str, llm, num_pairs: int = 3) -> list:
    """
    SINGLE LLM: Generate both questions AND answers in one call.
    Much faster and cheaper than dual-LLM approach.
    """
    prompt = f"""Generate exactly {num_pairs} Q&A pairs from this financial text.

Categories (cover all types):
- Hard Facts (numbers, dates, specific names): 1-2 pairs
- Strategic Summaries (business strategy, risks, opportunities): 1 pair
- Stylistic/Creative (tone, messaging, perspective): 0-1 pair

STRICT RULES:
1. ONLY use information explicitly stated in the context
2. Make questions specific and answerable
3. Answers must be factual and concise (1-3 sentences)
4. Return ONLY valid JSON, no markdown or extra text

Context:
{chunk_text}

Return ONLY this JSON format (no code blocks, no extra text):
[
  {{"question": "Q1?", "answer": "A1"}},
  {{"question": "Q2?", "answer": "A2"}},
  {{"question": "Q3?", "answer": "A3"}}
]"""
    
    try:
        response = llm.invoke(prompt)
        response_text = response.content if hasattr(response, 'content') else str(response)
        
        # Clean JSON (remove markdown code blocks if present)
        response_text = response_text.strip()
        if response_text.startswith('```json'):
            response_text = response_text.split('```json')[1].split('```')[0].strip()
        elif response_text.startswith('```'):
            response_text = response_text.split('```')[1].split('```')[0].strip()
        
        # Parse JSON
        pairs = json.loads(response_text)
        return pairs[:num_pairs]
        
    except json.JSONDecodeError as e:
        print(f"️  JSON parse error: {str(e)[:50]}")
        return []
    except Exception as e:
        print(f"️  Generation error: {str(e)[:50]}")
        return []

print("✅ Single LLM Q&A generation function ready")

✅ Single LLM Q&A generation function ready


In [6]:
# Test on sample chunk
test_chunk = chunks[10]['text']

print("🧪 Testing Q&A generation...\n")
print(f"Chunk text length: {len(test_chunk)} chars")
print("\nGenerating 3 Q&A pairs...\n")

test_pairs = generate_qa_pairs(test_chunk, llm, num_pairs=3)

if test_pairs:
    print(f'ffff {test_pairs}')
    print(f" Generated {len(test_pairs)} pairs\n")
    for i, pair in enumerate(test_pairs, 1):
        print(f"Pair {i}:")
        print(f"  Q: {pair['question']}")
        print(f"  A: {pair['answer'][:150]}...\n")
else:
    print(" No pairs generated - check API/JSON format")

🧪 Testing Q&A generation...

Chunk text length: 1320 chars

Generating 3 Q&A pairs...

ffff [{'question': 'What is the name of the company mentioned in the text?', 'answer': 'Uber Technologies, Inc.'}, {'question': 'What type of document is this report?', 'answer': 'Annual Report on Form 10-K'}, {'question': 'What is the purpose of the forward-looking statements in the report?', 'answer': "To disclose the company's plans, intentions, or expectations, but with no obligation to update them"}]
 Generated 3 pairs

Pair 1:
  Q: What is the name of the company mentioned in the text?
  A: Uber Technologies, Inc....

Pair 2:
  Q: What type of document is this report?
  A: Annual Report on Form 10-K...

Pair 3:
  Q: What is the purpose of the forward-looking statements in the report?
  A: To disclose the company's plans, intentions, or expectations, but with no obligation to update them...



## Step 3: Full Dataset Generation (ALL 482 CHUNKS)

**Using single LLM with 3 pairs per chunk:**
- Total chunks: 482
- Pairs per chunk: 3
- Expected total pairs: ~1,446
- Estimated API calls: ~482
- Time: ~30-45 minutes on Groq/Gemini
- Cost: FREE (both Groq and Gemini have free tiers)

In [7]:
# Configuration - OPTIMIZED FOR MAXIMUM DATA
max_chunks = len(chunks)  # ⭐ USE ALL 482 CHUNKS
qa_pairs_per_chunk = 3     # ⭐ 3 PAIRS PER CHUNK (realistic for 1500 chars)

print(f"Starting dataset generation...")
print(f"   Chunks to process: {max_chunks}")
print(f"   Q&A pairs per chunk: {qa_pairs_per_chunk}")
print(f"   Expected total pairs: {max_chunks * qa_pairs_per_chunk}")
print(f"   Estimated API calls: ~{max_chunks}")
print(f"   Estimated time: 30-45 minutes")
print(f"   Cost: FREE (Groq/Gemini)\n")

dataset = []
failed_chunks = []

def run_pairs(chunks):
    for i, chunk in enumerate(tqdm(chunks, desc="Generating Q&A")):
        chunk_text = chunk['text']
        chunk_id = chunk['chunk_id']

        # Generate Q&A pairs with SINGLE LLM
        pairs = generate_qa_pairs(chunk_text, llm, qa_pairs_per_chunk)

        if not pairs:
            failed_chunks.append(chunk_id)
            continue

        # Add to dataset
        for pair in pairs:
            dataset.append({
                'chunk_id': chunk_id,
                'question': pair['question'],
                'answer': pair['answer'],
                'context': chunk_text
            })

run_pairs(chunks)

# retry failed chunks
MAX_RETRIES = 3

for retry in range(MAX_RETRIES):

    if not failed_chunks:
        break

    print(f"Retry {retry+1}: {len(failed_chunks)} chunks")

    retry_ids = failed_chunks.copy()
    failed_chunks.clear()

    retry_chunks = [
        chunk for chunk in chunks
        if chunk["chunk_id"] in retry_ids
    ]

    run_pairs(retry_chunks)

print(f"\n Generated {len(dataset)} Q&A pairs")
if failed_chunks:
    print(f"️  Failed chunks: {len(failed_chunks)} (out of {max_chunks})")
    print(f"   Failed chunk IDs: {failed_chunks[:10]}..." if len(failed_chunks) > 10 else f"   Failed chunk IDs: {failed_chunks}")

Starting dataset generation...
   Chunks to process: 482
   Q&A pairs per chunk: 3
   Expected total pairs: 1446
   Estimated API calls: ~482
   Estimated time: 30-45 minutes
   Cost: FREE (Groq/Gemini)



Generating Q&A:  26%|██▌       | 123/482 [11:39<34:36,  5.79s/it]

️  JSON parse error: Expecting ',' delimiter: line 4 column 293 (char 6


Generating Q&A:  30%|███       | 147/482 [14:02<33:47,  6.05s/it]

️  JSON parse error: Expecting ',' delimiter: line 5 column 2 (char 401


...

Retry 2: 2 chunks


Generating Q&A: 100%|██████████| 2/2 [00:12<00:00,  6.37s/it]


️  JSON parse error: Expecting ',' delimiter: line 2 column 105 (char 1
Retry 3: 1 chunks


Generating Q&A: 100%|██████████| 1/1 [00:05<00:00,  5.59s/it]

️  JSON parse error: Expecting ',' delimiter: line 2 column 105 (char 1

 Generated 1443 Q&A pairs
️  Failed chunks: 1 (out of 482)
   Failed chunk IDs: [221]


## Step 4: Split and Save

In [8]:
import random

# Shuffle and split
random.seed(42)
random.shuffle(dataset)

split_idx = int(len(dataset) * 0.8)
train_data = dataset[:split_idx]
test_data = dataset[split_idx:]

print(f" Dataset split:")
print(f"   Train: {len(train_data)} (80%)")
print(f"   Test: {len(test_data)} (20%)")

# Save
artifacts_dir = Path(config['artifacts_root']) / "data_factory"
artifacts_dir.mkdir(parents=True, exist_ok=True)

with open(artifacts_dir / "train.jsonl", 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open(artifacts_dir / "golden_test_set.jsonl", 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')

print(f"\n Saved to {artifacts_dir}")
print(f"   - train.jsonl")
print(f"   - golden_test_set.jsonl")
print(f"\n Data Factory Complete!")

 Dataset split:
   Train: 1154 (80%)
   Test: 289 (20%)

 Saved to artifacts/data_factory
   - train.jsonl
   - golden_test_set.jsonl

 Data Factory Complete!
